# Support Vector Machines
Accompanies the notes on **Support Vector Machines**.

## 1. The three-point dual, checked

In [ ]:
import numpy as np
from sklearn.svm import SVC

X3 = np.array([[1, 1], [2, 0], [0, 0]]); y3 = np.array([1, 1, -1])
svm = SVC(kernel="linear", C=1e6).fit(X3, y3)            # huge C: practically a hard margin
print("w =", svm.coef_.round(4), " b =", svm.intercept_.round(4))
print("support vectors (row numbers):", svm.support_, " alpha_i * y_i:", svm.dual_coef_.round(4))
print("y_i (w.x_i + b):", (y3 * svm.decision_function(X3)).round(4))

## 2. Hinge loss vs logistic loss

In [ ]:
import matplotlib.pyplot as plt

m = np.linspace(-2, 3, 200)                               # margin value y * f(x)
plt.figure(figsize=(6, 3.5))
plt.plot(m, np.maximum(0, 1 - m), label="hinge (SVM)")
plt.plot(m, np.log2(1 + np.exp(-m)), label="logistic loss (in bits)")
plt.plot(m, (m < 0).astype(float), label="0-1 loss (misclassified)", ls="--")
plt.xlabel("y f(x)"); plt.ylabel("loss"); plt.legend(); plt.tight_layout(); plt.show()
for v in [-1, 0, 0.5, 1, 2]:
    print(f"y f(x) = {v:>4}: hinge {max(0, 1 - v):.2f}   logistic {np.log2(1 + np.exp(-v)):.3f}")

## 3. A kernel by hand

In [ ]:
x = np.array([1, 2]); z = np.array([3, 1])
phi = lambda v: np.array([v[0] ** 2, np.sqrt(2) * v[0] * v[1], v[1] ** 2])
print("phi(x).phi(z) =", round(phi(x) @ phi(z), 6), "  (x.z)^2 =", (x @ z) ** 2)

## 4. What C and gamma do

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

Xm, ym = make_moons(n_samples=400, noise=0.3, random_state=0)
xm_train, xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.5, random_state=0)
xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))
fig, axes = plt.subplots(3, 3, figsize=(11, 10))
for r, C in enumerate([0.1, 1, 100]):
    for c, gamma in enumerate([0.1, 1, 30]):
        model = SVC(kernel="rbf", C=C, gamma=gamma).fit(xm_train, ym_train)
        zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax = axes[r, c]
        ax.contourf(xx, yy, zz > 0, alpha=0.25, cmap="coolwarm"); ax.contour(xx, yy, zz, levels=[0], colors="k")
        ax.scatter(xm_train[:, 0], xm_train[:, 1], c=ym_train, s=8, cmap="coolwarm")
        ax.set_title(f"C={C}, gamma={gamma}: train {model.score(xm_train, ym_train):.2f}, test {model.score(xm_test, ym_test):.2f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        print(f"C={C:<5} gamma={gamma:<4}: train acc {model.score(xm_train, ym_train):.3f}  test acc {model.score(xm_test, ym_test):.3f}"
              f"  support vectors {len(model.support_)}")
plt.tight_layout(); plt.show()

## 5. RBF SVM on the bank data

In [ ]:
import time
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
numeric = bd.select_dtypes("number").columns.tolist()
X = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

pipe = Pipeline([("pre", ColumnTransformer([("num", StandardScaler(), numeric)], remainder="passthrough")),
                 ("svm", SVC(kernel="rbf"))])
grid = GridSearchCV(pipe, {"svm__C": [0.1, 1, 10, 100], "svm__gamma": [0.003, 0.01, 0.03, 0.1, 0.3]},
                    cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc", n_jobs=-1)
grid.fit(x_train, y_train)
print("best:", grid.best_params_, " CV AUC:", round(grid.best_score_, 4))
best = grid.best_estimator_
print("test AUC:", round(roc_auc_score(y_test, best.decision_function(x_test)), 4),
      " support vectors:", len(best[-1].support_), "of", len(x_train))

## 6. Training time vs number of rows

In [ ]:
for n in [1000, 2000, 4000, 6093]:
    start = time.time()
    best.fit(x_train.iloc[:n], y_train.iloc[:n])
    print(f"n = {n:5d}: fit time {time.time() - start:.2f}s")

## 7. Support vector regression on the loan data

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error

ld = pd.read_csv("../../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
cols = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months",
        "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
for col in cols:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
ld = ld[cols + ["Interest.Rate"]].dropna()
xr_train, xr_test, yr_train, yr_test = train_test_split(ld[cols], ld["Interest.Rate"], test_size=0.2, random_state=2)
for kernel, C in [("linear", 1), ("rbf", 1), ("rbf", 10), ("rbf", 100)]:
    svr = Pipeline([("scale", StandardScaler()), ("svr", SVR(kernel=kernel, C=C, epsilon=0.5))]).fit(xr_train, yr_train)
    print(f"SVR {kernel:6s} C={C:<4}: test MAE {mean_absolute_error(yr_test, svr.predict(xr_test)):.3f}, "
          f"support vectors {len(svr[-1].support_)} of {len(xr_train)}")

## Try it yourself
1. Add the point (3, 3) with label +1 in section 1.
2. `SVC(C=10, gamma=3)` on the moons.